In [3]:
import numpy as np

In [4]:
# define the constant values
NUM_OF_ITEMS = 500
NUM_OF_BINS = 10

In [5]:
class Ants:
    def __init__(self, num_of_items, num_of_bins, items):
        
        # initialize items
        self.items = items
        self.start_weights = np.random.rand(num_of_bins)
        self.middle_weights = np.random.rand(num_of_items-1, num_of_bins, num_of_bins)
        self.chosen = []
        self.difference = 99999999
        
    def normalize(self):
		
 		# normalize the weights
        self.start_weights = self.start_weights / np.sum(self.start_weights)
        self.middle_weights = self.middle_weights / np.sum(self.middle_weights, axis=1, keepdims=True)
        
    def forwardPass(self):
        
        # only allow function to be ran once for this class
        # if self.chosen != []:
        #     return
        
        # choose the first bin
        chosen_index = np.random.choice(np.arange(len(self.start_weights)), p=self.start_weights)
        self.chosen.append(chosen_index)
        
        # choose the next bin
        for weight_matrix in self.middle_weights:
            chosen_index = np.random.choice(np.arange(len(weight_matrix[chosen_index])), p=[sub_array[self.chosen[-1]] for sub_array in weight_matrix])
            self.chosen.append(chosen_index)
    
    def evaluate(self):
        # calculate the max difference between the bin that has the lowest number and highest number
        totals = [0 for i in range(NUM_OF_BINS)]
        for i in range(len(self.items)):
            totals[self.chosen[i]] += self.items[i]
        self.difference = max(totals) - min(totals)

In [6]:
ants = Ants(NUM_OF_ITEMS, NUM_OF_BINS, [i+1 for i in range(NUM_OF_ITEMS)])
ants.normalize()
ants.forwardPass()
ants.evaluate()
print(ants.difference)

8263


In [7]:
class Evolve:
    def __init__(self, size, iterations):
        self.population_size = size
        self.population = []
        self.iterations = iterations
        self.current_iteration = 1


    def selection(self):

        # store fitness of population
        fitness = []

        # evaluate the population
        for ant in self.population:
            ant.evaluate()
            fitness.append(ant.difference)
        
        # get n smallest
        best = np.partition(fitness, 1)[:2]
        return [self.population[fitness.index(i)] for i in best]


    # chat gpt generated this function
    def crossover(self, parent1, parent2):
        
        # Initialize two offspring with the same structure as the parents
        offspring1 = Ants(len(parent1.items), len(parent1.start_weights), parent1.items)
        offspring2 = Ants(len(parent2.items), len(parent2.start_weights), parent2.items)
        
        # Crossover for the start_weights (e.g., single-point crossover)
        crossover_point = np.random.randint(len(parent1.start_weights))
        offspring1.start_weights[:crossover_point] = parent1.start_weights[:crossover_point]
        offspring1.start_weights[crossover_point:] = parent2.start_weights[crossover_point:]
        
        offspring2.start_weights[:crossover_point] = parent2.start_weights[:crossover_point]
        offspring2.start_weights[crossover_point:] = parent1.start_weights[crossover_point:]
        
        # Crossover for the middle_weights (e.g., two-point crossover)
        for i in range(len(parent1.middle_weights)):
            crossover_point1 = np.random.randint(len(parent1.middle_weights[i]))
            crossover_point2 = np.random.randint(crossover_point1, len(parent1.middle_weights[i]))
            
            offspring1.middle_weights[i, :crossover_point1] = parent1.middle_weights[i, :crossover_point1]
            offspring1.middle_weights[i, crossover_point1:crossover_point2] = parent2.middle_weights[i, crossover_point1:crossover_point2]
            offspring1.middle_weights[i, crossover_point2:] = parent1.middle_weights[i, crossover_point2:]
            
            offspring2.middle_weights[i, :crossover_point1] = parent2.middle_weights[i, :crossover_point1]
            offspring2.middle_weights[i, crossover_point1:crossover_point2] = parent1.middle_weights[i, crossover_point1:crossover_point2]
            offspring2.middle_weights[i, crossover_point2:] = parent2.middle_weights[i, crossover_point2:]
        
        # Normalize weights to ensure they sum to 1
        offspring1.normalize()
        offspring2.normalize()
        
        offspring1.forwardPass()
        offspring2.forwardPass()
        
        return offspring1, offspring2
    
    
    # chat gpt generated this function
    def mutate(self, individual, mutation_rate=0.2):
        
        # Mutate start_weights
        for i in range(len(individual.start_weights)):
            if np.random.rand() < mutation_rate:
                # Add a small random value to the weight
                individual.start_weights[i] += np.random.normal(0, 0.1)  # Gaussian mutation
                # Ensure weight stays non-negative
                if individual.start_weights[i] < 0:
                    individual.start_weights[i] = 0
        
        # Mutate middle_weights
        for i in range(len(individual.middle_weights)):
            for j in range(len(individual.middle_weights[i])):
                for k in range(len(individual.middle_weights[i][j])):
                    if np.random.rand() < mutation_rate:
                        # Add a small random value to the weight
                        individual.middle_weights[i][j][k] += np.random.normal(0, 0.1)  # Gaussian mutation
                        # Ensure weight stays non-negative
                        if individual.middle_weights[i][j][k] < 0:
                            individual.middle_weights[i][j][k] = 0
        
        # Normalize the weights to ensure they still sum to 1
        individual.normalize()


    def replacement(self, offspring1, offspring2):
        
        # fitnesses of the population
        fitness = []

        # evaluate the population
        for ant in self.population:
            ant.evaluate()
            fitness.append(ant.difference)
            
        # get the index of the largest fitness
        worst = sorted(fitness)
        
        # replace the worst with the offspring
        self.population[fitness.index(worst[-2:][0])] = offspring1
        self.population[fitness.index(worst[-2:][1])] = offspring2
        


    def showFitnesses(self):
        fitnesses = []
        print(self.current_iteration, end="\t")
        # print the current population evaluation
        for ant in self.population:
            ant.evaluate()
            fitnesses.append(ant.difference)
            print(ant.difference, end="\t")
        
        print(f"AVG: {np.average(fitnesses)} MAX: {np.max(fitnesses)}")

    def start(self):
        
        self.current_iteration = 1

        # create the ant populuation
        for _ in range(self.population_size):
            ants = Ants(NUM_OF_ITEMS, NUM_OF_BINS, [i+1 for i in range(NUM_OF_ITEMS)])
            ants.normalize()
            ants.forwardPass()

            # append the population with this class
            self.population.append(ants)

        # run the iterations
        for i in range(self.iterations):
            
            # show the fitnesses in the current population
            self.showFitnesses()

            # perform selection
            select1, select2 = self.selection()
            
            # perform crossover
            offspring1, offspring2 = self.crossover(select1, select2)
            
            # perform mutation
            self.mutate(offspring1)
            self.mutate(offspring2)
            
            # replace into population
            self.replacement(offspring1, offspring2)
            
            self.current_iteration += 1

In [8]:
evolve = Evolve(10, 200)
evolve.start()

1	8299	7242	5387	6439	5350	4864	5926	6875	6375	4741	AVG: 6149.8 MAX: 8299
2	2415	4368	5387	6439	5350	4864	5926	6875	6375	4741	AVG: 5274.0 MAX: 6875
3	2415	4368	5387	6700	5350	4864	5926	4350	6375	4741	AVG: 5047.6 MAX: 6700
4	2415	4368	5387	5184	5350	4864	5926	4350	5882	4741	AVG: 4846.7 MAX: 5926
5	2415	4368	5387	5184	5350	4864	4570	4350	8640	4741	AVG: 4986.9 MAX: 8640
6	2415	4368	5074	5184	5350	4864	4570	4350	7878	4741	AVG: 4879.4 MAX: 7878
7	2415	4368	5074	5184	6076	4864	4570	4350	6932	4741	AVG: 4857.4 MAX: 6932
8	2415	4368	5074	5184	6718	4864	4570	4350	6632	4741	AVG: 4891.6 MAX: 6718
9	2415	4368	5074	5184	7516	4864	4570	4350	5990	4741	AVG: 4907.2 MAX: 7516
10	2415	4368	5074	5184	5075	4864	4570	4350	3761	4741	AVG: 4440.2 MAX: 5184
11	2415	4368	5074	5573	6284	4864	4570	4350	3761	4741	AVG: 4600.0 MAX: 6284
12	2415	4368	5074	8340	7598	4864	4570	4350	3761	4741	AVG: 5008.1 MAX: 8340
13	2415	4368	5074	7024	9720	4864	4570	4350	3761	4741	AVG: 5088.7 MAX: 9720
14	2415	4368	5074	6895	7527	4864	4